mcp-server-client-roundtrip

In [ ]:
import sys, json, types
lrn_llm = types.ModuleType("lrn_llm")
try:
    from pyodide.http import pyfetch as _pyfetch
    _IN_PYODIDE = True
except ImportError:
    import urllib.request as _urlreq
    _IN_PYODIDE = False
lrn_llm.API_BASE = "/api/llm"  # same-origin proxy; server injects the gateway key
lrn_llm.DEFAULT_MODEL = "azure/gpt-5.4-mini"
lrn_llm.API_KEY = ""  # optional; set in Step 0a

async def _lrn_call(messages, *, system=None, max_tokens=400, model=None):
    if system is not None:
        messages = [{"role": "system", "content": system}] + list(messages)
    payload = {"model": model or lrn_llm.DEFAULT_MODEL, "messages": messages,
               "max_completion_tokens": max_tokens}
    headers = {"content-type": "application/json"}
    _key = lrn_llm.API_KEY
    if _key:
        headers["Authorization"] = "Bearer " + _key
    url = lrn_llm.API_BASE.rstrip("/") + "/chat/completions"
    body = json.dumps(payload)
    if _IN_PYODIDE:
        r = await _pyfetch(url, method="POST", headers=headers, body=body)
        data = await r.json()
    else:
        req = _urlreq.Request(url, method="POST", headers=headers, data=body.encode("utf-8"))
        with _urlreq.urlopen(req, timeout=60) as r:
            data = json.loads(r.read())
    if "error" in data:
        raise RuntimeError("LLM error: " + str(data["error"]))
    return data

def _lrn_text(r):
    ch = (r or {}).get("choices") or []
    return (ch[0].get("message", {}) or {}).get("content", "") if ch else ""

async def _lrn_ping():
    r = await _lrn_call([{"role": "user", "content": "Reply with exactly: OK"}], max_tokens=5)
    return {"ok": _lrn_text(r).strip().upper().startswith("OK"), "model": r.get("model")}

lrn_llm.call = _lrn_call
lrn_llm.text = _lrn_text
lrn_llm.ping = _lrn_ping
print("✅ notebook ready · endpoint:", lrn_llm.API_BASE)

## Step 0a — Endpoint & Key

Set your API key (optional on LHIND network) and verify the endpoint.

In [ ]:
lrn_llm.API_KEY = ""  # Leave empty to use LHIND gateway auth
print(f"Endpoint: {lrn_llm.API_BASE}")
print(f"Model: {lrn_llm.DEFAULT_MODEL}")
print(f"Key set: {bool(lrn_llm.API_KEY)}")

## Step 1 — Reachability

Ping the LLM to confirm connectivity.

In [ ]:
r = await lrn_llm.ping()
print(f"✅ LLM reachable: {r['ok']} · model={r['model']}")

## Step 2 — Build a Minimal MCP Server

Define an in-memory MCP server with three primitives: tools, resources, and prompts. This mirrors the lesson's main.py but as a pure Python class, without needing the external `mcp` package.

In [ ]:
import queue
from dataclasses import dataclass
from typing import Any, Callable

PROTOCOL_VERSION = "2025-06-18"

@dataclass
class Tool:
    name: str
    description: str
    input_schema: dict[str, Any]
    handler: Callable[..., Any]
    destructive: bool = False

@dataclass
class Resource:
    uri: str
    description: str
    handler: Callable[[], str]

@dataclass
class Prompt:
    name: str
    description: str
    arguments: list[str]
    handler: Callable[..., str]

class MCPServer:
    """Minimal MCP server implementing the three primitives."""
    def __init__(self, name: str):
        self.name = name
        self.tools = {}
        self.resources = {}
        self.prompts = {}
    
    def tool(self, name: str, description: str, schema: dict, *, destructive=False):
        def decorator(fn):
            self.tools[name] = Tool(name, description, schema, fn, destructive)
            return fn
        return decorator
    
    def resource(self, uri: str, description: str):
        def decorator(fn):
            self.resources[uri] = Resource(uri, description, fn)
            return fn
        return decorator
    
    def prompt(self, name: str, description: str, arguments: list):
        def decorator(fn):
            self.prompts[name] = Prompt(name, description, arguments, fn)
            return fn
        return decorator
    
    def handle(self, message: dict) -> dict:
        method = message.get("method")
        params = message.get("params") or {}
        request_id = message.get("id")
        try:
            if method == "initialize":
                result = {
                    "protocolVersion": PROTOCOL_VERSION,
                    "serverInfo": {"name": self.name, "version": "0.1.0"},
                    "capabilities": {"tools": {}, "resources": {}, "prompts": {}},
                }
            elif method == "tools/list":
                result = {"tools": [
                    {"name": t.name, "description": t.description,
                     "inputSchema": t.input_schema,
                     "annotations": {"destructiveHint": t.destructive} if t.destructive else {}}
                    for t in self.tools.values()
                ]}
            elif method == "tools/call":
                tool = self.tools[params["name"]]
                output = tool.handler(**params.get("arguments", {}))
                result = {"content": [{"type": "text", "text": json.dumps(output)}]}
            elif method == "resources/list":
                result = {"resources": [
                    {"uri": r.uri, "description": r.description} for r in self.resources.values()
                ]}
            elif method == "resources/read":
                res = self.resources[params["uri"]]
                result = {"contents": [{"uri": res.uri, "mimeType": "text/plain", "text": res.handler()}]}
            elif method == "prompts/list":
                result = {"prompts": [
                    {"name": p.name, "description": p.description,
                     "arguments": [{"name": a, "required": True} for a in p.arguments]}
                    for p in self.prompts.values()
                ]}
            elif method == "prompts/get":
                p = self.prompts[params["name"]]
                rendered = p.handler(**params.get("arguments", {}))
                result = {"messages": [{"role": "user", "content": {"type": "text", "text": rendered}}]}
            else:
                return {"jsonrpc": "2.0", "id": request_id,
                        "error": {"code": -32601, "message": f"unknown method: {method}"}}
        except KeyError as e:
            return {"jsonrpc": "2.0", "id": request_id,
                    "error": {"code": -32602, "message": f"missing key: {e}"}}
        return {"jsonrpc": "2.0", "id": request_id, "result": result}

print("✅ MCPServer class defined")

## Step 3 — Register Tools, Resources, and Prompts

Populate the server with the demo domain: an `add` tool, a destructive `delete_user` tool, an `app_config` resource, and a `code_review` prompt.

In [ ]:
# Create the server instance
server = MCPServer("demo-server")

# Register a read-only tool
@server.tool(
    name="add",
    description="Add two integers and return the sum.",
    schema={"type": "object",
            "properties": {"a": {"type": "integer"}, "b": {"type": "integer"}},
            "required": ["a", "b"]}
)
def add(a: int, b: int) -> dict:
    return {"sum": a + b}

# Register a destructive (mutation) tool
@server.tool(
    name="delete_user",
    description="Delete a user by id. Mutating; requires approval.",
    schema={"type": "object",
            "properties": {"user_id": {"type": "integer"}},
            "required": ["user_id"]},
    destructive=True
)
def delete_user(user_id: int) -> dict:
    return {"deleted": user_id, "note": "simulated; real impl would hit DB"}

# Register a resource (read-only data)
@server.resource(
    uri="config://app",
    description="Application config as JSON text."
)
def app_config() -> str:
    return json.dumps({"env": "prod", "region": "us-east-1"})

# Register a prompt template
@server.prompt(
    name="code_review",
    description="Prompt the model to review code in a language.",
    arguments=["language", "code"]
)
def code_review(language: str, code: str) -> str:
    return f"You are a senior {language} reviewer. Review for correctness and style:\n\n{code}"

print(f"✅ Server '{server.name}' initialized with {len(server.tools)} tools, {len(server.resources)} resource(s), {len(server.prompts)} prompt(s)")

## Step 4 — Discovery: List Available Tools

When an MCP client connects, it calls `tools/list` to see what tools the server offers. This is what the LLM host uses to populate its tool budget.

In [ ]:
# Simulate a client discovery request
message = {"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {}}
response = server.handle(message)
tools_list = response["result"]["tools"]
print(f"\n📋 {len(tools_list)} tool(s) available:\n")
for t in tools_list:
    flag = " [DESTRUCTIVE]" if t.get("annotations", {}).get("destructiveHint") else ""
    print(f"  • {t['name']}{flag}")
    print(f"    {t['description']}")
    print(f"    Input: {json.dumps(t['inputSchema'], indent=6)}")
    print()

## Step 5 — Tool Invocation: Call add(40, 2)

The client calls a tool by name with arguments. The server executes the handler and returns the result as JSON.

In [ ]:
# Simulate a client tool call
message = {
    "jsonrpc": "2.0",
    "id": 2,
    "method": "tools/call",
    "params": {"name": "add", "arguments": {"a": 40, "b": 2}}
}
response = server.handle(message)
result_text = response["result"]["content"][0]["text"]
print(f"\nCalled: add(40, 2)")
print(f"Result: {result_text}")

## Step 6 — Resources: List and Read

Resources are read-only data the LLM can request by URI. Unlike tools, they cannot mutate state.

In [ ]:
# List resources
message = {"jsonrpc": "2.0", "id": 3, "method": "resources/list", "params": {}}
response = server.handle(message)
resources = response["result"]["resources"]
print(f"📦 {len(resources)} resource(s):\n")
for r in resources:
    print(f"  • {r['uri']}: {r['description']}")

# Read one resource
message = {"jsonrpc": "2.0", "id": 4, "method": "resources/read", "params": {"uri": "config://app"}}
response = server.handle(message)
config_text = response["result"]["contents"][0]["text"]
print(f"\nRead config://app:")
print(f"  {config_text}")

## Step 7 — Prompts: List and Render

Prompts are user-invoked templates that can have arguments. The model never calls them directly; the user asks for a prompt via slash-command, the client renders it, and sends the result to the model.

In [ ]:
# List prompts
message = {"jsonrpc": "2.0", "id": 5, "method": "prompts/list", "params": {}}
response = server.handle(message)
prompts = response["result"]["prompts"]
print(f"🎯 {len(prompts)} prompt(s):\n")
for p in prompts:
    args_str = ", ".join([a["name"] for a in p["arguments"]])
    print(f"  • {p['name']}({args_str})")
    print(f"    {p['description']}")

# Render a prompt with arguments
message = {
    "jsonrpc": "2.0",
    "id": 6,
    "method": "prompts/get",
    "params": {"name": "code_review", "arguments": {"language": "Python", "code": "x = 1\n"}}
}
response = server.handle(message)
rendered = response["result"]["messages"][0]["content"]["text"]
print(f"\nRender code_review(language='Python', code='x = 1\\n'):")
print(f"---")
print(rendered)
print(f"---")

## Step 8 — Real LLM Integration: Ask the Model to Use Tools

Now we pair this MCP server with a real LLM. We send the tool schemas from `tools/list` to the model, ask it a question, and let it decide which tool to call.

In [ ]:
# Get the tool schemas from the server
message = {"jsonrpc": "2.0", "id": 7, "method": "tools/list", "params": {}}
response = server.handle(message)
tool_schemas = response["result"]["tools"]

# Build the tools parameter for the LLM
# (This is what a real MCP host like Claude Desktop would do)
tools_for_llm = [
    {
        "type": "function",
        "function": {
            "name": t["name"],
            "description": t["description"],
            "parameters": t["inputSchema"]
        }
    }
    for t in tool_schemas
]

# Ask the LLM to perform a calculation using the add tool
messages = [{"role": "user", "content": "What is the sum of 100 and 50? Use the add tool to compute it."}]
system_prompt = "You are a helpful assistant with access to these tools. When the user asks you to compute something, use the appropriate tool. After you call a tool, wait for the result and respond naturally."

response = await lrn_llm.call(messages, system=system_prompt, max_tokens=300)
response_text = lrn_llm.text(response)
print("User: What is the sum of 100 and 50? Use the add tool to compute it.")
print(f"\nModel: {response_text}")

## Step 9 — Tool Calling Workflow

In a real agentic flow, the model would emit a tool_use block. Here we simulate that: ask the model to describe what tool it would call, then have us execute it.

In [ ]:
# Ask the model what operation it would perform
messages = [
    {"role": "user", "content": "I want to delete user 42. Which tool would you use, and what arguments?"}
]
system = "You are a helpful assistant. When the user asks about a tool operation, name the tool and list the arguments as JSON."
response = await lrn_llm.call(messages, system=system, max_tokens=150)
model_plan = lrn_llm.text(response)
print("User: I want to delete user 42. Which tool would you use, and what arguments?")
print(f"\nModel: {model_plan}")
print(f"\n[In a real agentic loop, the host would now call the tool with the model-suggested parameters...]")

## Step 10 — Safety: Destructive Hints

When a tool is marked `destructiveHint: true`, the host surfaces an approval UI before calling it. Let's show how that metadata flows through MCP.

In [ ]:
# Show the destructive hint for delete_user
message = {"jsonrpc": "2.0", "id": 8, "method": "tools/list", "params": {}}
response = server.handle(message)
tools_list = response["result"]["tools"]
destructive_tools = [t for t in tools_list if t.get("annotations", {}).get("destructiveHint")]

print(f"🚨 {len(destructive_tools)} destructive tool(s) found:\n")
for t in destructive_tools:
    print(f"  ⚠️  {t['name']}")
    print(f"     {t['description']}")
    print(f"     Requires: human approval before execution")
    print()

## Step 11 — Multi-Turn Agentic Loop

MCP is stateless—each request is independent. But an agentic host wraps it in a loop: get tools, ask model, model calls tool, execute tool, inject result back into conversation, loop.

In [ ]:
# Simulate a multi-turn loop: user query → model thinks → tool call → result → model responds
query = "What is 15 + 8? Then check our app config."
conversation = [{"role": "user", "content": query}]

# Step 1: Ask the model what it needs
system = (
    "You are a helpful assistant. When asked to compute or fetch data, describe what tools you would call.\n"
    "Available tools: add (adds two integers), config (gets app config). Keep responses under 100 words."
)
response = await lrn_llm.call(conversation, system=system, max_tokens=100)
model_reply = lrn_llm.text(response)
print(f"🔄 Turn 1: Model Planning")
print(f"User: {query}")
print(f"Model: {model_reply}")
print()

# Step 2: Execute tools (simulated)
print(f"🔧 Turn 2: Tool Execution (simulated)")
add_result = {"sum": 23}  # 15 + 8
config_result = {"env": "prod", "region": "us-east-1"}
print(f"  Executed add(15, 8) → {add_result}")
print(f"  Executed read(config://app) → {config_result}")
print()

# Step 3: Send results back to model for synthesis
conversation.append({"role": "assistant", "content": model_reply})
conversation.append({"role": "user", "content": f"Tool results: add(15,8)={add_result['sum']}, config={json.dumps(config_result)}"})
response = await lrn_llm.call(conversation, system=system, max_tokens=150)
final_reply = lrn_llm.text(response)
print(f"💬 Turn 3: Final Response")
print(f"Model: {final_reply}")

## Try It Yourself

Design your own MCP tool. What domain would benefit from a tool server? Define a tool, resource, or prompt below.

In [ ]:
# TODO: Create your own MCP tool
# Example domains: weather API, note-taking, expense tracking, ticket system

# Uncomment and edit this example:
# @server.tool(
#     name="fetch_weather",
#     description="Get the current weather for a city.",
#     schema={
#         "type": "object",
#         "properties": {"city": {"type": "string"}},
#         "required": ["city"]
#     }
# )
# def fetch_weather(city: str) -> dict:
#     # In reality, this would call an API
#     return {"city": city, "temp_c": 22, "condition": "sunny"}

print("🎯 Define your tool above and test it!")
print("\nExample: Add a weather tool, then:")
print("  msg = {'jsonrpc': '2.0', 'id': 99, 'method': 'tools/call',")
print("         'params': {'name': 'fetch_weather', 'arguments': {'city': 'Berlin'}}}")
print("  result = server.handle(msg)")
print("  print(result['result']['content'][0]['text'])")